In [2]:
# API 키, 구글계정 및 앱패스워드는 직접 입력 금지, .env파일로 저장

# ============================
# 1. 기본 설정
# ============================
import os
import sqlite3
import smtplib
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from openai import OpenAI

REMINDER_DAYS = [30, 14, 7, 5, 3, 1]
REMINDER_DAYS2 = [5,2,1]


load_dotenv("reminder_info.env", override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

GMAIL_ADDRESS = os.getenv("GMAIL_ADDRESS")             # 본인의 GMAIL 주소 호출
GMAIL_APP_PASSWORD = os.getenv("GMAIL_APP_PASSWORD")   # 본인의 앱패스워드 호출 (실제 비밀번호가 아님에 유의)



#1-1 액셀파일 불러오기, DB 경로설정

EXCEL_PATH = "schedule.xlsx"
DB_PATH = "reminder_log.db"

# 1-2 오픈AI API 설정 및 호출

client = OpenAI(api_key=OPENAI_API_KEY)


# 2. 함수정의: 로그 DB 생성
# ============================


def init_db():

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS send_log(
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            event_id TEXT,
            reminder_day INTEGER,
            recipient TEXT,
            subject TEXT,
            sent_at TEXT,
            status TEXT
        )
    """)

    conn.commit()
    conn.close()


# ============================
# 3. (함수정의) 이미 발송했는지 확인
# ============================

def already_sent(event_id, reminder_day):

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT COUNT(*)
        FROM send_log
        WHERE event_id = ?
          AND reminder_day = ?
          AND status = 'SUCCESS'
    """, (event_id, reminder_day))

    count = cursor.fetchone()[0]

    conn.close()

    return count > 0


# 4. GPT로 메일 생성
# ============================

def generate_email(event, days_left):
    event_date = event["기준일"].strftime("%Y년 %m월 %d일")
    prompt = f"""
너는 정부 R&D 사업 운영 담당자를 지원하는 업무비서야.

아래 일정을 담당자에게 알려주는 내부 업무메일을 다음과 같이 작성해줘.

[일정정보]
사업명: {event["사업명"]}
일정유형: {event["일정유형"]}
일정명: {event["일정명"]}
일정일: {event_date}
남은기간: D-{days_left}
참고사항: {event["참고사항"]}

작성원칙:
- 전문기관이 연구수행기관 담당자에게 메일을 보내는 것처럼 정중하게 작성할 것. (500자 이내)
- 첫 줄은 반드시 '제목: 간결한 메일 제목'으로 작성
- 다음 줄에 '본문:'을 작성
- 사업명, 일정유형, 일정명, 일정일, 남은기간, 참고사항은 본문 맨 위에 제시하고, 그 아래에 안내문 작성
- 마크다운 표나 HTML 태그 없이 일반 텍스트로 작성
- 입력된 정보에 없는 사실을 임의로 만들지 말 것
- 불확실한 것은 '확인 필요'라고 표시
- 아래 형식으로 작성

제목: ...
본문:
...
"""
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt,
        extra_headers={"Accept-Encoding": "identity"}
    )

    text = response.output_text.strip()

    # 아주 단순한 제목/본문 분리
    if "본문:" in text:
        subject_part, body = text.split("본문:", 1)
        subject = subject_part.replace("제목:", "").strip()
    else:
        subject = f"[D-{days_left}] {event['사업명']} {event['일정명']} 안내"
        body = text

    return subject, body.strip()


# ============================
# 5. Gmail 발송
# ============================


def send_email(recipient, subject, body):

    message = MIMEMultipart()
    message["From"] = GMAIL_ADDRESS
    message["To"] = recipient
    message["Subject"] = subject

    message.attach(
        MIMEText(body, "plain", "utf-8")
    )


    with smtplib.SMTP_SSL(
        "smtp.gmail.com",
        465
    ) as server:

        server.login(
            GMAIL_ADDRESS,
            GMAIL_APP_PASSWORD
        )

        server.send_message(message)



# ============================
# 6. 로그 저장
# ============================

def save_log(
    event_id,
    reminder_day,
    recipient,
    subject,
    status
):

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO send_log (
            event_id,
            reminder_day,
            recipient,
            subject,
            sent_at,
            status
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        event_id,
        reminder_day,
        recipient,
        subject,
        datetime.now().isoformat(),
        status
    ))

    conn.commit()
    conn.close()



# ============================
# 7. 메인 실행
# ============================

def run_reminder():

    df = pd.read_excel(EXCEL_PATH)

    df["기준일"] = pd.to_datetime(df["기준일"])

    today = pd.Timestamp.today().normalize()

    for _, event in df.iterrows():

        if str(event["활성화"]).upper() != "Y":
            continue

        event_date = event["기준일"].normalize()

        days_left = (
            event_date - today
        ).days

        # 알림 대상 날짜인가?
        if days_left not in REMINDER_DAYS:
            continue

        event_id = str(event["event_id"])

        # 이미 보냈다면 중복 발송 방지
        if already_sent(
            event_id,
            days_left
        ):
            continue

        recipient = event["수신자"]

        try:

            subject, body = generate_email(
                event,
                days_left
            )

            send_email(
                recipient,
                subject,
                body
            )

            save_log(
                event_id,
                days_left,
                recipient,
                subject,
                "SUCCESS"
            )

            print(
                f"{event_id} D-{days_left} 발송 완료"
            )

        except Exception as e:

            save_log(
                event_id,
                days_left,
                recipient,
                "",
                f"ERROR: {str(e)}"
            )

            print(
                f"{event_id} 오류:",
                e
            )


# ============================
# 프로그램 실행
# ============================

if __name__ == "__main__":

    init_db()
    run_reminder()



In [2]:
import sys
print(sys.executable)

C:\Users\21009\Anaconda3\python.exe


In [7]:
"""지정 날짜·시각에 엑셀 일정 한 건을 메일로 보내고 종료합니다.
DB, 로그 파일, D-day 발송 필터, 중복 발송 방지 기능은 없습니다.
실행 중에는 PC와 파이썬을 켜두세요. PC 시각(한국 시간 설정)을 사용합니다.
"""
import os
import time
import ssl
from pathlib import Path
from datetime import datetime
import smtplib
import pandas as pd
from dotenv import load_dotenv
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from openai import OpenAI

# ===== 아래 세 항목을 수정하세요 =====
SEND_AT = "2026-10-30 00:55:30"  # 원하는 발송 날짜·시각 (미래 시각)
RECIPIENT = "kmh@koita.or.kr"  # 본인의 실제 사내메일 주소
EVENT_ROW = 0  # schedule.xlsx의 첫 번째 데이터 행. 두 번째는 1

# .py는 파일이 있는 폴더, 주피터에 붙여 넣으면 현재 작업 폴더 기준
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
EXCEL_PATH = BASE_DIR / "schedule.xlsx"
load_dotenv(BASE_DIR / "reminder_info.env", override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")  # 기존에 성공한 모델 사용
GMAIL_ADDRESS = os.getenv("GMAIL_ADDRESS", "").strip()
GMAIL_APP_PASSWORD = os.getenv("GMAIL_APP_PASSWORD", "").replace(" ", "").strip()

def generate_email(event, days_left):
    event_date = event["기준일"].strftime("%Y년 %m월 %d일")
    prompt = f"""
너는 정부 R&D 사업 운영 담당자를 지원하는 업무비서야.

아래 일정을 담당자에게 알려주는 내부 업무메일을 다음과 같이 작성해줘.

[일정정보]
사업명: {event["사업명"]}
일정유형: {event["일정유형"]}
일정명: {event["일정명"]}
일정일: {event_date}
남은기간: D-{days_left}
참고사항: {event["참고사항"]}

작성원칙:
- 전문기관이 연구수행기관 담당자에게 메일을 보내는 것처럼 정중하게 작성할 것. (500자 이내)
- 첫 줄은 반드시 '제목: 간결한 메일 제목'으로 작성
- 다음 줄에 '본문:'을 작성
- 사업명, 일정유형, 일정명, 일정일, 남은기간, 참고사항은 본문 맨 위에 제시하고, 그 아래에 안내문 작성
- 마크다운 표나 HTML 태그 없이 일반 텍스트로 작성
- 입력된 정보에 없는 사실을 임의로 만들지 말 것
- 불확실한 것은 '확인 필요'라고 표시
- 아래 형식으로 작성

제목: ...
본문:
...
"""
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt,
        extra_headers={"Accept-Encoding": "identity"}
    )

    text = response.output_text.strip()

    # 아주 단순한 제목/본문 분리
    if "본문:" in text:
        subject_part, body = text.split("본문:", 1)
        subject = subject_part.replace("제목:", "").strip()
    else:
        subject = f"[D-{days_left}] {event['사업명']} {event['일정명']} 안내"
        body = text

    return subject, body.strip()


def send_email(recipient, subject, body):

    message = MIMEMultipart()
    message["From"] = GMAIL_ADDRESS
    message["To"] = recipient
    message["Subject"] = subject

    message.attach(
        MIMEText(body, "plain", "utf-8")
    )


    with smtplib.SMTP_SSL(
        "smtp.gmail.com",
        465,
        context=ssl.create_default_context(),
        timeout=30
    ) as server:

        server.login(
            GMAIL_ADDRESS,
            GMAIL_APP_PASSWORD
        )

        server.send_message(message)


def run_reminder():
    print("프로그램 시작", flush=True)
    send_at = datetime.strptime(SEND_AT, "%Y-%m-%d %H:%M:%S")
    if send_at <= datetime.now():
        raise ValueError("SEND_AT을 현재보다 미래의 날짜·시각으로 바꿔주세요.")
    if RECIPIENT == "본인주소@회사도메인.co.kr":
        raise ValueError("RECIPIENT를 실제 사내메일 주소로 바꿔주세요.")
    if not all([OPENAI_API_KEY, GMAIL_ADDRESS, GMAIL_APP_PASSWORD]):
        raise ValueError("reminder_info.env의 API 키·Gmail 주소·앱 비밀번호를 확인하세요.")

    df = pd.read_excel(EXCEL_PATH)
    if not 0 <= EVENT_ROW < len(df):
        raise ValueError("엑셀에 선택한 데이터 행이 없습니다.")
    event = df.iloc[EVENT_ROW].copy()
    event["기준일"] = pd.to_datetime(event["기준일"])
    if pd.isna(event["기준일"]):
        raise ValueError("선택한 일정의 기준일이 비어 있습니다.")
    if pd.isna(event["참고사항"]):
        event["참고사항"] = "없음"
    days_left = (event["기준일"].date() - send_at.date()).days
    if days_left < 0:
        raise ValueError("기준일이 발송일보다 과거입니다. 엑셀 기준일을 확인하세요.")

    # 생성 시간 때문에 발송이 늦어지는 것을 줄이기 위해 미리 작성합니다.
    global client
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("GPT 메일 생성 중...", flush=True)
    subject, body = generate_email(event, days_left)
    print("수신자:", RECIPIENT, flush=True)
    print("제목:", subject, flush=True)
    print("예약 발송 시각:", send_at, flush=True)

    if datetime.now() >= send_at:
        print("메일 생성 중 예약 시각이 지나 지금 전송합니다.", flush=True)
    while True:
        remaining = (send_at - datetime.now()).total_seconds()
        if remaining <= 0:
            break
        time.sleep(min(1, remaining))

    print("Gmail 전송 시작:", datetime.now(), flush=True)
    send_email(RECIPIENT, subject, body)
    print("메일 전송 완료:", datetime.now(), flush=True)


if __name__ == "__main__":
    run_reminder()


프로그램 시작
GPT 메일 생성 중...
수신자: kmh@koita.or.kr
제목: K-HERO 육성·지원 선정평가 일정 안내
예약 발송 시각: 2026-10-30 00:55:30
Gmail 전송 시작: 2026-10-30 00:55:30.012356
메일 전송 완료: 2026-10-30 00:55:32.884637


In [1]:
"""정부사업 리마인더 — DB 및 로그 저장 없는 MVP 버전.
실행할 때마다 schedule.xlsx의 알림 대상 일정을 발송하고 종료합니다.
중복 발송 방지가 없으므로 같은 날 재실행하면 같은 메일이 다시 발송됩니다.
콘솔 print 출력만 유지하며 로그 파일이나 DB는 생성하지 않습니다.
"""


import os
import smtplib
import ssl
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from openai import OpenAI

# .py에서는 파일 폴더, 주피터에 붙여 넣으면 현재 작업 폴더 기준
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
load_dotenv(BASE_DIR / "reminder_info.env", override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

GMAIL_ADDRESS = os.getenv("GMAIL_ADDRESS", "").strip()
GMAIL_APP_PASSWORD = os.getenv("GMAIL_APP_PASSWORD", "").replace(" ", "").strip()

REMINDER_DAYS = [30, 14, 7, 5, 3, 1]
EXCEL_PATH = BASE_DIR / "schedule.xlsx"
client = OpenAI(api_key=OPENAI_API_KEY)



# 1. GPT 메일 생성 (기존 방식 유지)
def generate_email(event, days_left):
    event_date = event["기준일"].strftime("%Y년 %m월 %d일")
    prompt = f"""
너는 정부 R&D 사업 운영 담당자를 지원하는 업무비서야.

아래 일정을 담당자에게 알려주는 내부 업무메일을 다음과 같이 작성해줘.

[일정정보]
사업명: {event["사업명"]}
일정유형: {event["일정유형"]}
일정명: {event["일정명"]}
일정일: {event_date}
남은기간: D-{days_left}
참고사항: {event["참고사항"]}

작성원칙:
- 전문기관이 연구수행기관 담당자에게 메일을 보내는 것처럼 정중하게 작성할 것. (500자 이내)
- 첫 줄은 반드시 '제목: 간결한 메일 제목'으로 작성
- 다음 줄에 '본문:'을 작성
- 사업명, 일정유형, 일정명, 일정일, 남은기간, 참고사항은 본문 맨 위에 제시하고, 그 아래에 안내문 작성
- 마크다운 표나 HTML 태그 없이 일반 텍스트로 작성
- 입력된 정보에 없는 사실을 임의로 만들지 말 것
- 불확실한 것은 '확인 필요'라고 표시
- 아래 형식으로 작성

제목: ...
본문:
...
"""
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prompt,
        extra_headers={"Accept-Encoding": "identity"}
    )

    text = response.output_text.strip()

    # 아주 단순한 제목/본문 분리
    if "본문:" in text:
        subject_part, body = text.split("본문:", 1)
        subject = subject_part.replace("제목:", "").strip()
    else:
        subject = f"[D-{days_left}] {event['사업명']} {event['일정명']} 안내"
        body = text

    return subject, body.strip()


# 2. Gmail 발송
def send_email(recipient, subject, body):

    message = MIMEMultipart()
    message["From"] = GMAIL_ADDRESS
    message["To"] = recipient
    message["Subject"] = subject

    message.attach(
        MIMEText(body, "plain", "utf-8")
    )


    with smtplib.SMTP_SSL(
        "smtp.gmail.com",
        465,
        context=ssl.create_default_context(),
        timeout=30
    ) as server:

        server.login(
            GMAIL_ADDRESS,
            GMAIL_APP_PASSWORD
        )

        server.send_message(message)


# 3. 엑셀 확인 및 메일 발송

def run_reminder():
    today = pd.Timestamp.today().normalize()
    print(f"일정 확인 시작: {today.date()}", flush=True)
    print(f"엑셀 파일: {EXCEL_PATH}", flush=True)

    df = pd.read_excel(EXCEL_PATH)
    df["기준일"] = pd.to_datetime(df["기준일"])
    print(f"전체 일정: {len(df)}건", flush=True)
    sent_count = 0
    failed_count = 0
    skipped_count = 0

    for _, event in df.iterrows():
        event_id = str(event["event_id"])

        if str(event["활성화"]).strip().upper() != "Y":
            print(f"{event_id}: 비활성 일정 → 제외", flush=True)
            skipped_count += 1
            continue

        if pd.isna(event["기준일"]):
            print(f"{event_id}: 기준일이 비어 있음 → 제외", flush=True)
            skipped_count += 1
            continue

        days_left = (event["기준일"].normalize() - today).days
        if days_left not in REMINDER_DAYS:
            print(f"{event_id}: 남은 일수 {days_left}일 → 알림 대상 아님", flush=True)
            skipped_count += 1
            continue

        try:
            if pd.isna(event["수신자"]) or not str(event["수신자"]).strip():
                raise ValueError("수신자 주소가 비어 있습니다.")
            recipient = str(event["수신자"]).strip()
            if pd.isna(event["참고사항"]):
                event = event.copy()
                event["참고사항"] = "없음"

            print(f"{event_id}: D-{days_left} 메일 생성 중", flush=True)
            subject, body = generate_email(event, days_left)
            send_email(recipient, subject, body)
            sent_count += 1
            print(f"{event_id}: D-{days_left} 전송 완료 → {recipient}", flush=True)

        except Exception as e:
            failed_count += 1
            print(f"{event_id}: 오류 — {type(e).__name__}: {e}", flush=True)

    print(
        f"일정 확인 완료: 전송 {sent_count}건 / 오류 {failed_count}건 / 제외 {skipped_count}건",
        flush=True
    )
    if failed_count:
        raise RuntimeError(f"메일 처리 중 {failed_count}건의 오류가 발생했습니다.")


if __name__ == "__main__":
    run_reminder()


일정 확인 시작: 2026-10-30
엑셀 파일: C:\Users\21009\Desktop\AITF\Project1\schedule.xlsx
전체 일정: 9건
RND001: 남은 일수 51일 → 알림 대상 아님
RND002: 남은 일수 -1일 → 알림 대상 아님
RND003: 남은 일수 62일 → 알림 대상 아님
RND004: 남은 일수 89일 → 알림 대상 아님
RND005: 남은 일수 122일 → 알림 대상 아님
RND006: 남은 일수 62일 → 알림 대상 아님
RND007: 남은 일수 94일 → 알림 대상 아님
RND008: 남은 일수 28일 → 알림 대상 아님
TEST_NIGHT: D-1 메일 생성 중
TEST_NIGHT: D-1 전송 완료 → kmh@koita.or.kr
일정 확인 완료: 전송 1건 / 오류 0건 / 제외 8건


In [2]:
import sys
print(sys.executable)

C:\Users\21009\Anaconda3\python.exe
